In [1]:
import os
import sys
import json
import time
import glob
import math
import yaml
import random
import wandb
import numpy as np
import numpy.typing as npt
import torch
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter
from cs336_basics.data import get_batch, load, save
from cs336_basics.modules.layers import TransformerLM
from cs336_basics.modules.loss import CrossEntropyLoss
from cs336_basics.modules.activation import GLU, Softmax
from cs336_basics.modules.optimizer import SGD, AdamW, compute_lr, gradient_cliping
from cs336_basics.tokenizer import Tokenizer
from tokenizers import Tokenizer as HFTokenizer

#### Helpers / config

In [2]:
print("--- Loading Configuration ---")
with open('config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

model_args = cfg['model_args']
training_args = cfg['training_args']
data_args = cfg['data_args']

os.makedirs(data_args['checkpoint_dir'], exist_ok=True)

device = torch.device(training_args.get('device', 'cuda') if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# H100 111
if device.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

# reproducibility (optional)
seed = training_args.get('seed', 42)
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if device.type == 'cuda':
    torch.cuda.manual_seed_all(seed)
    # optional performance flags
    if training_args.get('cudnn_benchmark', True):
        torch.backends.cudnn.benchmark = True
    if training_args.get('cudnn_deterministic', False):
        torch.backends.cudnn.deterministic = True

--- Loading Configuration ---
Using device: cuda


#### Tokenizer / data

In [3]:
hf_tokenizer = HFTokenizer.from_file(data_args['vocab_path'])

tokenizer = Tokenizer(
    vocab={token: idx for token, idx in hf_tokenizer.get_vocab().items()},
    merges=data_args['merges_path']
)

model_args['vocab_size'] = len(tokenizer.vocab)
print(f"Tokenizer loaded. Vocab size: {model_args['vocab_size']}")

Tokenizer loaded. Vocab size: 10000


#### Model init utils

In [4]:
def _init_weights(module):
    # classic initialization used in many transformer implementations
    if isinstance(module, (nn.Linear, nn.Conv1d, nn.Conv2d)):
        # small normal
        nn.init.normal_(module.weight, mean=0.0, std=training_args.get('init_std', 0.02))
        if getattr(module, 'bias', None) is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        nn.init.normal_(module.weight, mean=0.0, std=training_args.get('init_std', 0.02))
    elif isinstance(module, nn.LayerNorm):
        # LayerNorm: weight ones, bias zeros
        if getattr(module, 'weight', None) is not None:
            nn.init.ones_(module.weight)
        if getattr(module, 'bias', None) is not None:
            nn.init.zeros_(module.bias)

def residual_projection_rescale(model, num_layers):
    # scale residual projection weights (heuristic used in nanoGPT / others)
    # multiply std by 1/sqrt(2*num_layers) for parameters named like .*c_proj.weight
    factor = math.sqrt(2.0 * max(1, num_layers))
    for name, p in model.named_parameters():
        if name.endswith('c_proj.weight') or '.c_proj.' in name:
            # If currently normal-initialized, rescale
            with torch.no_grad():
                if p.dtype.is_floating_point:
                    p.mul_(1.0 / factor)

def try_weight_tying(model):
    """Try to tie embedding and lm_head weights if both exist."""
    # Common names: token_embedding, tok_embeddings, wte, embed_tokens ; lm_head, head, lm_head
    emb = None
    head = None
    for attr in ('token_embedding', 'tok_embeddings', 'wte', 'embed_tokens', 'embedding'):
        if hasattr(model, attr):
            emb = getattr(model, attr)
            break
    # find head (linear)
    for attr in ('lm_head', 'head', 'output_head', 'decoder'):
        if hasattr(model, attr):
            head = getattr(model, attr)
            break
    # If we didn't find, try scanning
    if emb is None:
        for n, m in model.named_modules():
            if isinstance(m, nn.Embedding):
                emb = m
                break
    if head is None:
        for n, m in model.named_modules():
            if isinstance(m, nn.Linear):
                # pick last linear as head candidate (best effort)
                head = m

    if isinstance(emb, nn.Embedding) and isinstance(head, nn.Linear):
        try:
            head_weight = getattr(head, 'weight', None)
            if head_weight is not None and emb.weight.shape == head_weight.shape:
                emb.weight = head.weight
                print("Weight tying applied: embedding <-> lm_head")
            else:
                # if shapes differ, try transposed tying if head expects vocab x hidden and embedding hidden x vocab
                if emb.weight.shape[1] == head.weight.shape[0] and emb.weight.shape[0] == head.weight.shape[1]:
                    # do not enforce but try to tie via reference transpose not supported -> skip
                    print("Found embedding and head but shapes do not match for direct tying; skipping tying.")
        except Exception as e:
            print("Weight tying failed (ignored):", e)

#### Build model (no compile yet)

In [5]:
print("--- Building model (uncompiled) ---")
model = TransformerLM(**model_args).to(device)

# apply explicit init
model.apply(_init_weights)

# try residual scaling if possible
num_layers = None
# try a few heuristics to get number of layers
if hasattr(model, 'num_layers'):
    num_layers = getattr(model, 'num_layers')
elif hasattr(model, 'n_layer'):
    num_layers = getattr(model, 'n_layer')
elif 'num_layers' in model_args:
    num_layers = model_args['num_layers']
elif 'n_layer' in model_args:
    num_layers = model_args['n_layer']
else:
    # fallback: try counting blocks module named 'blocks' or 'layers'
    for name, module in model.named_children():
        if name in ('blocks', 'layers', 'transformer_blocks'):
            num_layers = sum(1 for _ in module.children())
            break

if num_layers is None:
    num_layers = training_args.get('num_layers', 12)  # safe default

residual_projection_rescale(model, num_layers)
try_weight_tying(model)

# number of params (safe)
num_params = sum(p.numel() for p in model.parameters())
print(f"Model created with {num_params:,} parameters.")

--- Building model (uncompiled) ---
Model created with 22,696,448 parameters.


#### Optimizer & loss

In [6]:
optimizer = AdamW(model.parameters(), lr=training_args['learning_rate'])
loss_fn = CrossEntropyLoss()

#### Checkpoint resume

In [7]:
start_iter = 0
if data_args.get('resume_from_checkpoint'):
    ckpt_path = data_args['resume_from_checkpoint']
    print(f"Resuming training from explicit checkpoint: {ckpt_path}")
    start_iter = load(ckpt_path, model, optimizer) or 0

# also check checkpoint dir for latest
ckpts = glob.glob(os.path.join(data_args['checkpoint_dir'], "model_iter_*.pt"))
if ckpts:
    latest_ckpt = max(ckpts, key=os.path.getctime)
    print(f"Found latest checkpoint: {latest_ckpt}")
    start_iter = load(latest_ckpt, model, optimizer) or start_iter
else:
    if start_iter == 0:
        print("From Scratch Training")

From Scratch Training


#### Compile (after loading)

In [8]:
if training_args.get('use_torch_compile', True) and hasattr(torch, 'compile'):
    try:
        print("Compiling model with torch.compile() ...")
        model = torch.compile(model)
    except Exception as e:
        print("torch.compile() failed or not beneficial; continuing without compile:", e)

Compiling model with torch.compile() ...


In [9]:
# H100 Mix
use_amp = training_args.get('use_amp', True) and device.type == 'cuda'
amp_dtype = torch.bfloat16 if use_amp else None
if use_amp:
    print("Enabled Mixed Precision (AMP) with torch.bfloat16 for Hopper Architecture.")
amp_context = torch.amp.autocast('cuda', dtype=amp_dtype) if use_amp else torch.autocast('cpu', enabled=False)

Enabled Mixed Precision (AMP) with torch.bfloat16 for Hopper Architecture.


#### Data (memmap)

In [10]:
print("--- Loading Data with np.memmap ---")
train_data = np.memmap(data_args['train_data_path'], dtype=np.uint16, mode='r')
valid_data = np.memmap(data_args['valid_data_path'], dtype=np.uint16, mode='r')
print(f"Train tokens: {len(train_data):,}, Val tokens: {len(valid_data):,}")

--- Loading Data with np.memmap ---
Train tokens: 3,038,773,016, Val tokens: 74,004,384


#### Logging: wandb + tb

In [11]:
use_wandb = training_args.get('use_wandb', True)
if use_wandb:
    wandb.init(
        entity="lzq666amn-github",
        project=training_args.get('wandb_project', 'llm_project'),
        name=training_args.get('wandb_run_name', None),
        config={
            **model_args, **training_args, **data_args
        }
    )
writer = SummaryWriter(log_dir=training_args.get('tensorboard_logdir', "runs/llm_run"))

#### Evaluate function

In [12]:
@torch.no_grad()
def evaluate(iter_num):
    model.eval()
    valid_loss = 0.0
    if iter_num < 500:
        eval_iters = min(10, training_args.get('eval_max_iters', 20))
    elif iter_num < 5000:
        eval_iters = min(50, training_args.get('eval_max_iters', 100))
    else:
        eval_iters = training_args.get('eval_max_iters', 100)
        
    for _ in range(eval_iters):
        x, y = get_batch(valid_data, training_args['batch_size'], model_args['context_length'], device)
        # H100 1111
        with amp_context:
            logits = model(x)
            loss = loss_fn(logits.view(-1, model_args['vocab_size']), y.view(-1))
        valid_loss += loss.item()
    model.train()
    
    avg_loss = valid_loss / max(1, eval_iters)
    perplexity = math.exp(avg_loss)
    return avg_loss, perplexity

#### Training loop

In [ ]:
grad_accum_steps = training_args.get('grad_accum_steps', 1)
max_iters = training_args['max_iters']
t0 = time.time()

print("--- Starting Training Loop ---")
train_losses = []
val_losses = []
iterations = []
eval_iterations = []

for iter_num in range(start_iter, max_iters):
    lr = compute_lr(
        iter_num,
        training_args['learning_rate'],
        training_args.get('min_lr', 0.0),
        training_args.get('warmup_steps', 100),
        training_args.get('lr_decay_steps', max_iters)
    )
    for pg in optimizer.param_groups:
        pg['lr'] = lr

    optimizer.zero_grad(set_to_none=True)
    micro_loss = 0.0
    for micro_step in range(grad_accum_steps):
        inputs, targets = get_batch(train_data, training_args['batch_size'], model_args['context_length'], device)

        with amp_context:
            logits = model(inputs)
            loss = loss_fn(logits.view(-1, model_args['vocab_size']), targets.view(-1))
            loss = loss / grad_accum_steps
            
        loss.backward()
        micro_loss += loss.item()

    total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), training_args['gradient_clip_val'])

    optimizer.step()

    iter_loss = micro_loss * grad_accum_steps
    train_ppl = math.exp(iter_loss)
    
    train_losses.append(iter_loss)
    iterations.append(iter_num)

    writer.add_scalar("Loss/train", iter_loss, iter_num)
    writer.add_scalar("Perplexity/train", train_ppl, iter_num)
    writer.add_scalar("LR", lr, iter_num)
    writer.add_scalar("GradNorm/total", total_norm, iter_num)
    
    if use_wandb:
        wandb.log({
            "train/loss": iter_loss, 
            "train/perplexity": train_ppl,
            "lr": lr, 
            "grad_norm": total_norm, 
            "iter": iter_num
        }, step=iter_num)

    if iter_num % training_args.get('log_interval', 10) == 0:
        t1 = time.time()
        print(
            f"Iter {iter_num}/{max_iters}, "
            f"Train Loss: {iter_loss:.4f}, "
            f"Train PPL: {train_ppl:.2f}, "
            f"LR: {lr:.6f}, "
            f"GradNorm: {total_norm:.4f}, "
            f"Time per {training_args.get('log_interval',10)} iters: {(t1-t0)*1000:.1f}ms"
        )
        t0 = t1

    if iter_num > 0 and iter_num % training_args.get('eval_interval', 1000) == 0:
        val_loss, val_ppl = evaluate(iter_num)
        val_losses.append(val_loss)
        eval_iterations.append(iter_num)

        writer.add_scalar("Loss/val", val_loss, iter_num)
        writer.add_scalar("Perplexity/val", val_ppl, iter_num)
        
        if use_wandb:
            wandb.log({"val/loss": val_loss, "val/perplexity": val_ppl, "iter": iter_num}, step=iter_num)

        print(f"--- Eval at iter {iter_num}: Val Loss: {val_loss:.4f} | Val PPL: {val_ppl:.2f} ---")
        checkpoint_path = os.path.join(data_args['checkpoint_dir'], f"model_iter_{iter_num}.pt")
        
        try:
            save(model, optimizer, iter_num, checkpoint_path)
        except Exception:
            torch.save({
                'iter': iter_num,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'rng_state': torch.get_rng_state(),
            }, checkpoint_path)
            print("Saved fallback checkpoint.")

writer.close()
if use_wandb:
    wandb.finish()
print("Training complete! Logs written to", training_args.get('tensorboard_logdir', "runs/llm_run"))

--- Starting Training Loop ---


W0320 10:02:55.402000 137854791554176 torch/fx/experimental/symbolic_shapes.py:4449] [0/0] xindex is not in var_ranges, defaulting to unknown range.
W0320 10:02:59.391000 137838007936576 torch/fx/experimental/symbolic_shapes.py:4449] [0/0] xindex is not in var_ranges, defaulting to unknown range.


Iter 0/50000, Train Loss: 9.2543, Train PPL: 10449.35, LR: 0.000000, GradNorm: 0.3789, Time per 10 iters: 11842.9ms
Iter 10/50000, Train Loss: 9.2255, Train PPL: 10152.66, LR: 0.000015, GradNorm: 0.3826, Time per 10 iters: 2127.5ms
Iter 20/50000, Train Loss: 9.1253, Train PPL: 9184.65, LR: 0.000030, GradNorm: 0.4382, Time per 10 iters: 3053.5ms
Iter 30/50000, Train Loss: 8.9181, Train PPL: 7465.74, LR: 0.000045, GradNorm: 0.7398, Time per 10 iters: 2200.8ms
Iter 40/50000, Train Loss: 8.4958, Train PPL: 4894.05, LR: 0.000060, GradNorm: 1.2933, Time per 10 iters: 2442.6ms
Iter 50/50000, Train Loss: 8.0774, Train PPL: 3220.96, LR: 0.000075, GradNorm: 1.3795, Time per 10 iters: 1762.8ms
Iter 60/50000, Train Loss: 7.7175, Train PPL: 2247.41, LR: 0.000090, GradNorm: 1.2220, Time per 10 iters: 1453.2ms
Iter 70/50000, Train Loss: 7.4234, Train PPL: 1674.66, LR: 0.000105, GradNorm: 0.8858, Time per 10 iters: 1660.4ms
Iter 80/50000, Train Loss: 7.2653, Train PPL: 1429.87, LR: 0.000120, GradNorm:

W0320 10:06:53.424000 137854791554176 torch/fx/experimental/symbolic_shapes.py:4449] [0/1] xindex is not in var_ranges, defaulting to unknown range.


--- Eval at iter 1000: Val Loss: 4.4714 | Val PPL: 87.48 ---
save checkpoints/model_iter_1000.pt iterations: 1000
Iter 1010/50000, Train Loss: 4.4743, Train PPL: 87.73, LR: 0.000299, GradNorm: 0.2716, Time per 10 iters: 15527.0ms
Iter 1020/50000, Train Loss: 4.4688, Train PPL: 87.26, LR: 0.000299, GradNorm: 0.2833, Time per 10 iters: 3186.8ms
Iter 1030/50000, Train Loss: 4.4817, Train PPL: 88.38, LR: 0.000299, GradNorm: 0.2768, Time per 10 iters: 3161.3ms
Iter 1040/50000, Train Loss: 4.4546, Train PPL: 86.02, LR: 0.000299, GradNorm: 0.2531, Time per 10 iters: 2554.5ms
Iter 1050/50000, Train Loss: 4.4519, Train PPL: 85.79, LR: 0.000299, GradNorm: 0.2564, Time per 10 iters: 1697.8ms
Iter 1060/50000, Train Loss: 4.4539, Train PPL: 85.97, LR: 0.000299, GradNorm: 0.3690, Time per 10 iters: 1644.9ms
Iter 1070/50000, Train Loss: 4.4492, Train PPL: 85.56, LR: 0.000299, GradNorm: 0.2554, Time per 10 iters: 2663.7ms
Iter 1080/50000, Train Loss: 4.4309, Train PPL: 84.01, LR: 0.000299, GradNorm: 0